# RoBERTa — Validation Predictions & Confusion Matrix

Loads the fine-tuned **XLM-RoBERTa-large** checkpoint, predicts on the held-out
validation split (same seed / stratification used in `ensemble-kaggle.ipynb`),
and reports:

1. Accuracy + competition score `1 - MAE/4`
2. Per-class precision/recall/F1 (`classification_report` heatmap)
3. Confusion matrix — raw counts **and** row-normalised side-by-side

All paths are Kaggle paths matching the other notebooks in this repo.
Adjust the **Config** cell if your dataset/model slugs differ.

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ── Config ───────────────────────────────────────────────────────────────────
DATA_DIR        = '/kaggle/input/datasets/vladcvx/sentiment-classification-data-256/data1'
ENC_CACHE_VAL   = '/kaggle/input/datasets/vladcvx/sentiment-classification-data-256-roberta/data1/roberta-encodings/roberta_val_encodings.pt'

MODEL_NAME      = 'xlm-roberta-large'
MODEL_PATH      = '/kaggle/input/models/vladcvx/cil-roberta-1-epoch/pytorch/default/1/roberta_best_model_init_num_epochs-5lr-1e-06step_size-Nonegamma-Nonetrain_size-226800val_size-25200.pth'

MAX_LENGTH      = 256
BATCH_SIZE      = 16
VAL_FRACTION    = 0.10

PLOTS_DIR       = '/kaggle/working/roberta_cm_plots'
OUT_DIR         = '/kaggle/working'
os.makedirs(PLOTS_DIR, exist_ok=True)

LABEL_NAMES     = ['1\u2605', '2\u2605', '3\u2605', '4\u2605', '5\u2605']
CLASS_COLORS    = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']

## 1. Load data + recreate the same train/val split

In [ ]:
def clean_text(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip()

train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
train_df['sentence'] = train_df['sentence'].apply(clean_text)

train_sentences, val_sentences, train_labels, val_labels = train_test_split(
    train_df['sentence'].tolist(),
    train_df['label'].tolist(),
    test_size=VAL_FRACTION,
    random_state=SEED,
    stratify=train_df['label'].tolist(),
)
val_labels = np.asarray(val_labels)
print(f'Train: {len(train_sentences):,} | Val: {len(val_sentences):,}')

## 2. Tokenize the validation set (or load cached encodings)

In [ ]:
class ReviewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: v[idx].clone() for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if os.path.exists(ENC_CACHE_VAL):
    print(f'Loading cached encodings from {ENC_CACHE_VAL}')
    val_enc = torch.load(ENC_CACHE_VAL, weights_only=False)
else:
    print('Cache miss \u2014 tokenizing on the fly\u2026')
    val_enc = tokenizer(
        val_sentences,
        truncation=True, padding=True,
        max_length=MAX_LENGTH, return_tensors='pt',
    )

val_dataset = ReviewsDataset(val_enc, val_labels.tolist())
val_loader  = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
)
print(f'Val batches: {len(val_loader)}')

## 3. Load the fine-tuned RoBERTa checkpoint

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)
state = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state)
model = model.to(device).eval()
print('RoBERTa loaded.')

## 4. Predict on the validation set

In [ ]:
@torch.no_grad()
def predict_probs(model, dataloader):
    all_probs = []
    for batch in tqdm(dataloader, desc='RoBERTa val'):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        batch.pop('labels', None)
        logits = model(**batch).logits
        all_probs.append(F.softmax(logits, dim=1).cpu().numpy())
    return np.vstack(all_probs)  # (N, 5)

val_probs = predict_probs(model, val_loader)
val_preds = val_probs.argmax(axis=1)
print(f'val_probs: {val_probs.shape} | val_preds: {val_preds.shape}')

## 5. Metrics — accuracy, score, classification report

In [ ]:
def evaluation_score(preds, labels):
    """Competition metric: 1 - MAE/4."""
    return 1.0 - np.mean(np.abs(np.asarray(preds) - np.asarray(labels))) / 4.0

acc      = accuracy_score(val_labels, val_preds)
score    = evaluation_score(val_preds, val_labels)
f1_macro = f1_score(val_labels, val_preds, average='macro')

print(f'Accuracy        : {acc:.4f}')
print(f'Score (1-MAE/4) : {score:.4f}')
print(f'Macro F1        : {f1_macro:.4f}\n')
print(classification_report(val_labels, val_preds, target_names=LABEL_NAMES, digits=4))

In [ ]:
report_dict = classification_report(
    val_labels, val_preds, target_names=LABEL_NAMES, output_dict=True, digits=4,
)
report_df = pd.DataFrame(report_dict).T
display_cols = ['precision', 'recall', 'f1-score']

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(
    report_df.loc[LABEL_NAMES, display_cols],
    annot=True, fmt='.3f', cmap='Blues', cbar=False, ax=ax,
)
ax.set_title('RoBERTa \u2014 classification report (val)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/roberta_classification_report.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Confusion matrix — raw counts and row-normalised

In [ ]:
def plot_cm(preds, labels, title, ax, normalize=False):
    cm = confusion_matrix(labels, preds, labels=list(range(5)))
    if normalize:
        cm = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    sns.heatmap(
        cm, annot=True,
        fmt='.2f' if normalize else 'd',
        cmap='Blues', ax=ax,
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
        vmin=0, vmax=1 if normalize else None,
        linewidths=0.5, cbar=False,
    )
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_cm(val_preds, val_labels,
        f'RoBERTa \u2014 counts  (acc={acc:.4f}, score={score:.4f})',
        axes[0], normalize=False)
plot_cm(val_preds, val_labels,
        'RoBERTa \u2014 row-normalised (recall per class)',
        axes[1], normalize=True)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/roberta_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Most confused class pairs

Same diagnostic as in `ensemble_level_analysis.ipynb` \u2014 useful for the report.

In [ ]:
wrong_mask  = val_preds != val_labels
wrong_true  = val_labels[wrong_mask]
wrong_pred  = val_preds[wrong_mask]

pair_counts = {}
for t, p in zip(wrong_true, wrong_pred):
    pair_counts[(int(t), int(p))] = pair_counts.get((int(t), int(p)), 0) + 1

top_pairs    = sorted(pair_counts.items(), key=lambda x: -x[1])[:10]
total_errors = int(wrong_mask.sum())

print(f'Total errors: {total_errors:,} / {len(val_labels):,} '
      f'({total_errors / len(val_labels):.2%})\n')
print(f'{"True":>6}  {"Pred":>6}  {"Count":>8}  {"% of errors":>12}')
print('-' * 40)
for (t, p), cnt in top_pairs:
    print(f'{LABEL_NAMES[t]:>6}  {LABEL_NAMES[p]:>6}  {cnt:>8,}  '
          f'{cnt / total_errors * 100:>11.1f}%')

## 8. Save artifacts

Stored under `/kaggle/working` so they're easy to grab from the right pane.

In [ ]:
np.save(f'{OUT_DIR}/roberta_probs_val.npy', val_probs)
np.save(f'{OUT_DIR}/roberta_preds_val.npy', val_preds)

pd.DataFrame({
    'true_label' : val_labels,
    'pred_label' : val_preds,
    'sentence'   : val_sentences,
}).to_csv(f'{OUT_DIR}/roberta_val_predictions.csv', index=False)

print('Saved:')
print(f'  {OUT_DIR}/roberta_probs_val.npy   {val_probs.shape}')
print(f'  {OUT_DIR}/roberta_preds_val.npy   {val_preds.shape}')
print(f'  {OUT_DIR}/roberta_val_predictions.csv')
print(f'  {PLOTS_DIR}/roberta_confusion_matrix.png')
print(f'  {PLOTS_DIR}/roberta_classification_report.png')